# Deliverable 2 — Delta Lakehouse Implementation for Food Delivery Analytics

## Project Overview

This notebook implements the Delta Lakehouse component of a food delivery data pipeline using **Apache Spark** and **Delta Lake**.

The solution follows the **Medallion Architecture**:

- **Bronze layer:** stores raw delivery records exactly as received.
- **Silver layer:** cleans and standardizes operational data.
- **Gold layer:** produces aggregated business metrics for analytics and reporting.

## Delta Lake Features Demonstrated

- Delta table creation
- Bronze, Silver, and Gold architecture
- Schema enforcement
- Real `MERGE` upsert using `delivery_id` as the business key
- Business-level aggregation
- Automated verification through displayed outputs and a rejected invalid write

## Pipeline Flow

Raw delivery records  
→ Bronze Delta table  
→ Cleaning and standardization  
→ Silver Delta table  
→ MERGE updates and inserts  
→ Schema-enforcement test  
→ Gold business aggregates


## 1. Environment Setup

This section installs the versions of PySpark and Delta Lake used by the notebook.

- **PySpark 3.5.0** provides the distributed processing engine.
- **Delta Spark 3.2.0** adds ACID transactions, schema enforcement, table versioning, and `MERGE` support.

Pinning exact versions helps make the notebook reproducible across Colab sessions.


In [1]:
# PySpark provides the distributed data processing engine.
# Delta Lake extends Spark by adding ACID transactions, schema enforcement, versioning, and MERGE operations.

!pip install pyspark==3.5.0 delta-spark==3.2.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 16.4 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.0-py2.py3-none-any.whl size=317425346 sha256=f3c768645281df04d4e1fefed4d74e27169d8c7fa657412d5760c70e8aca9b5c
  Stored in directory: /root/.cache/pip/wheels/84/40/20/65eefe766118e0a8f8e385cc3ed6e9eb7241c7e51cfc04c51a
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.3
    Uninstalling pyspark-4.0.3:
      Successfully uninstalled pyspark-4.0.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-conn

## 2. Imports, Spark Configuration, and Data Contract

This section:

1. Imports the required Spark and Delta Lake libraries.
2. Creates a Spark session configured with Delta Lake extensions.
3. Defines the delivery-record schema used throughout the pipeline.
4. Provides a small formatting helper for readable notebook output.

The explicit schema acts as a data contract by fixing column names, data types, and nullability before data is written to the lakehouse.


In [2]:
import os
import shutil

from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StructField,
    StructType,
    StringType,
)


# ─────────────────────────────────────────────────────────────────────────────
# SESSION SETUP
# ─────────────────────────────────────────────────────────────────────────────

def create_spark_session() -> SparkSession:
    """Create and return a local Spark session configured for Delta Lake.

    The Delta SQL extension enables Delta-specific commands, while the
    Delta catalog allows Spark to read and write Delta tables correctly.
    """
    builder = (
        SparkSession.builder
        .appName("RestaurantDeliveryLakehouse")
        .master("local[*]")
        .config(
            "spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension"
        )
        .config(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog"
        )
    )

    return configure_spark_with_delta_pip(builder).getOrCreate()


# Explicit schema used for all valid delivery batches.
# Defining the schema prevents Spark from inferring inconsistent types.
DELIVERY_SCHEMA = StructType([
    StructField("delivery_id", StringType(), nullable=False),
    StructField("customer_id", StringType(), nullable=True),
    StructField("restaurant_name", StringType(), nullable=True),
    StructField("order_amount", DoubleType(), nullable=True),
    StructField("delivery_minutes", IntegerType(), nullable=True),
    StructField("city_zone", StringType(), nullable=True),
    StructField("delivery_status", StringType(), nullable=True),
])


def sep(label: str) -> None:
    """Print a visible section separator in notebook output."""
    print(f"\n{'=' * 65}")
    print(f"  {label}")
    print("=" * 65)


## 3. End-to-End Delta Lakehouse Pipeline

The main pipeline performs five stages:

1. **Bronze:** write raw delivery records to Delta Lake.
2. **Silver:** clean and standardize text fields.
3. **MERGE:** update an existing delivery and insert a new delivery using `delivery_id`.
4. **Schema enforcement:** attempt an invalid append with an unexpected column and confirm that Delta Lake rejects it.
5. **Gold:** calculate genuine business aggregates by city zone.

The table paths are deleted at the start of each run so the notebook produces a clean and repeatable demonstration.


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# MAIN PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

def main() -> None:
    """Run the complete Bronze, Silver, MERGE, enforcement, and Gold pipeline."""

    # Local Delta-table locations for each Medallion layer.
    BRONZE_PATH = "./delivery_data/bronze"
    SILVER_PATH = "./delivery_data/silver"
    GOLD_PATH = "./delivery_data/gold"

    # Remove previous data so every execution begins with a clean environment.
    if os.path.exists("./delivery_data"):
        shutil.rmtree("./delivery_data")

    # Start Spark only after the working directory has been reset.
    spark = create_spark_session()
    spark.sparkContext.setLogLevel("ERROR")


    # ─────────────────────────────────────────────────────────────────────────
    # 1. BRONZE LAYER
    # ─────────────────────────────────────────────────────────────────────────

    sep("1. BRONZE LAYER")

    # Initial raw delivery batch. Mixed capitalization and whitespace are
    # intentionally included so the Silver transformations are visible.
    batch_1 = [
        (
            "DEL001",
            "CUS001",
            "Pizza Palace",
            85.50,
            28,
            " North ",
            "COMPLETED"
        ),
        (
            "DEL002",
            "CUS002",
            "Burger House",
            42.00,
            18,
            "East",
            "Completed"
        ),
        (
            "DEL003",
            "CUS003",
            "Sushi Corner",
            110.25,
            35,
            "West",
            "PREPARING"
        ),
        (
            "DEL004",
            "CUS004",
            "Pasta Villa",
            63.75,
            25,
            "South",
            "Completed"
        ),
        (
            "DEL005",
            "CUS005",
            "Taco Spot",
            38.50,
            20,
            "North",
            "OUT FOR DELIVERY"
        ),
        (
            "DEL006",
            "CUS006",
            "Burger House",
            55.00,
            22,
            "East",
            "COMPLETED"
        ),
    ]

    # Apply the explicit delivery schema before writing raw data.
    bronze_df = spark.createDataFrame(
        batch_1,
        DELIVERY_SCHEMA
    )

    # Persist the raw batch in Delta format as the Bronze layer.
    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .save(BRONZE_PATH)
    )

    print("Raw records stored in the Bronze Delta table:")

    spark.read \
        .format("delta") \
        .load(BRONZE_PATH) \
        .show(truncate=False)


    # ─────────────────────────────────────────────────────────────────────────
    # 2. SILVER LAYER
    # ─────────────────────────────────────────────────────────────────────────

    sep("2. SILVER LAYER")

    # Read Bronze back from storage before applying transformations.
    bronze_table = (
        spark.read
        .format("delta")
        .load(BRONZE_PATH)
    )

    # Standardize fields while preserving the original business values.
    silver_df = (
        bronze_table
        .withColumn(
            "restaurant_name",
            F.trim(F.col("restaurant_name"))
        )
        .withColumn(
            "city_zone",
            F.upper(F.trim(F.col("city_zone")))
        )
        .withColumn(
            "delivery_status",
            F.lower(F.trim(F.col("delivery_status")))
        )
    )

    # Store the cleaned result as the Silver Delta table.
    (
        silver_df.write
        .format("delta")
        .mode("overwrite")
        .save(SILVER_PATH)
    )

    print("Cleaned and standardized records stored in Silver:")

    spark.read \
        .format("delta") \
        .load(SILVER_PATH) \
        .orderBy("delivery_id") \
        .show(truncate=False)


    # ─────────────────────────────────────────────────────────────────────────
    # 3. DELTA MERGE / UPSERT
    # ─────────────────────────────────────────────────────────────────────────

    sep("3. DELTA MERGE / UPSERT")

    # Second batch demonstrates both MERGE paths:
    # DEL003 already exists and will be updated; DEL007 is new and will be inserted.
    batch_2 = [
        (
            "DEL003",
            "CUS003",
            "Sushi Corner",
            110.25,
            40,
            "West",
            "Completed"
        ),
        (
            "DEL007",
            "CUS007",
            "Grill Station",
            72.00,
            30,
            "South",
            "Out for Delivery"
        ),
    ]

    updates_df = spark.createDataFrame(
        batch_2,
        DELIVERY_SCHEMA
    )

    # Apply the same Silver standardization rules to incoming updates.
    cleaned_updates_df = (
        updates_df
        .withColumn(
            "restaurant_name",
            F.trim(F.col("restaurant_name"))
        )
        .withColumn(
            "city_zone",
            F.upper(F.trim(F.col("city_zone")))
        )
        .withColumn(
            "delivery_status",
            F.lower(F.trim(F.col("delivery_status")))
        )
    )

    # Open the Silver path as a DeltaTable so the transactional MERGE API is available.
    silver_table = DeltaTable.forPath(
        spark,
        SILVER_PATH
    )

    # Upsert on the delivery_id business key.
    (
        silver_table.alias("target")
        .merge(
            cleaned_updates_df.alias("source"),
            "target.delivery_id = source.delivery_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("Silver table after MERGE:")

    spark.read \
        .format("delta") \
        .load(SILVER_PATH) \
        .orderBy("delivery_id") \
        .show(truncate=False)



    # ─────────────────────────────────────────────────────────────────────────
    # 4. SCHEMA ENFORCEMENT
    # ─────────────────────────────────────────────────────────────────────────

    sep("4. SCHEMA ENFORCEMENT")

    # This schema intentionally adds an unexpected driver_rating column.
    # Appending it without schema evolution should be rejected by Delta Lake.
    invalid_schema = StructType([
        StructField(
            "delivery_id",
            StringType(),
            nullable=False
        ),
        StructField(
            "customer_id",
            StringType(),
            nullable=True
        ),
        StructField(
            "restaurant_name",
            StringType(),
            nullable=True
        ),
        StructField(
            "order_amount",
            DoubleType(),
            nullable=True
        ),
        StructField(
            "delivery_minutes",
            IntegerType(),
            nullable=True
        ),
        StructField(
            "city_zone",
            StringType(),
            nullable=True
        ),
        StructField(
            "delivery_status",
            StringType(),
            nullable=True
        ),
        StructField(
            "driver_rating",
            DoubleType(),
            nullable=True
        ),
    ])

    invalid_batch = [
        (
            "DEL999",
            "CUS999",
            "Invalid Restaurant",
            50.00,
            15,
            "North",
            "Completed",
            4.8
        )
    ]

    invalid_df = spark.createDataFrame(
        invalid_batch,
        invalid_schema
    )

    # The failed append is the expected outcome and proves schema enforcement.
    try:
        (
            invalid_df.write
            .format("delta")
            .mode("append")
            .save(SILVER_PATH)
        )

        print("Unexpected result: invalid schema was accepted.")

    except Exception as error:
        print("Schema enforcement successfully rejected the write.")
        print("Rejection reason:")
        print(str(error).splitlines()[0])



    # ─────────────────────────────────────────────────────────────────────────
    # 5. GOLD LAYER
    # ─────────────────────────────────────────────────────────────────────────

    sep("5. GOLD LAYER")

    # Read the latest Silver snapshot after the MERGE.
    current_silver_df = (
        spark.read
        .format("delta")
        .load(SILVER_PATH)
    )

    # Produce genuine business aggregates grouped by city zone.
    gold_df = (
        current_silver_df
        .groupBy("city_zone")
        .agg(
            F.count("delivery_id").alias(
                "total_deliveries"
            ),
            F.round(
                F.sum("order_amount"),
                2
            ).alias(
                "total_order_value"
            ),
            F.round(
                F.avg("delivery_minutes"),
                2
            ).alias(
                "average_delivery_minutes"
            ),
            F.sum(
                F.when(
                    F.col("delivery_status") == "completed",
                    1
                ).otherwise(0)
            ).alias(
                "completed_deliveries"
            )
        )
        .orderBy("city_zone")
    )

    # Persist the aggregate as the Gold Delta table.
    (
        gold_df.write
        .format("delta")
        .mode("overwrite")
        .save(GOLD_PATH)
    )

    print("Aggregated delivery analytics stored in Gold:")

    spark.read \
        .format("delta") \
        .load(GOLD_PATH) \
        .orderBy("city_zone") \
        .show(truncate=False)



    sep("DELTA LAKEHOUSE COMPLETED")

    print("Bronze table path:", BRONZE_PATH)
    print("Silver table path:", SILVER_PATH)
    print("Gold table path:", GOLD_PATH)

    # Release local Spark resources after all verification output is produced.
    spark.stop()

if __name__ == "__main__":
    main()


  1. BRONZE LAYER
Raw records stored in the Bronze Delta table:
+-----------+-----------+---------------+------------+----------------+---------+----------------+
|delivery_id|customer_id|restaurant_name|order_amount|delivery_minutes|city_zone|delivery_status |
+-----------+-----------+---------------+------------+----------------+---------+----------------+
|DEL004     |CUS004     |Pasta Villa    |63.75       |25              |South    |Completed       |
|DEL005     |CUS005     |Taco Spot      |38.5        |20              |North    |OUT FOR DELIVERY|
|DEL006     |CUS006     |Burger House   |55.0        |22              |East     |COMPLETED       |
|DEL001     |CUS001     |Pizza Palace   |85.5        |28              | North   |COMPLETED       |
|DEL002     |CUS002     |Burger House   |42.0        |18              |East     |Completed       |
|DEL003     |CUS003     |Sushi Corner   |110.25      |35              |West     |PREPARING       |
+-----------+-----------+---------------+---

# Technical Documentation

## Architecture

The notebook uses a local Delta Lakehouse with three storage layers:

- `./delivery_data/bronze`
- `./delivery_data/silver`
- `./delivery_data/gold`

Each layer is stored in Delta format rather than plain Parquet.

## Business Key

`delivery_id` is used as the business key during the Delta `MERGE`.

- Matching records are updated.
- New records are inserted.

This makes the operation a true upsert rather than a simple append.

## Transformation Rules

The Silver layer applies the following standardization:

- Trims whitespace from `restaurant_name`
- Trims and uppercases `city_zone`
- Trims and lowercases `delivery_status`

## Schema-Enforcement Test

The notebook creates an invalid DataFrame containing an extra `driver_rating` column and attempts to append it to the Silver Delta table.

The expected behavior is failure. The captured rejection message proves that schema enforcement is active.

## Gold Aggregation

The Gold layer groups data by `city_zone` and calculates:

- Total deliveries
- Total order value
- Average delivery time
- Completed deliveries

This is a genuine aggregate and not a copy of Silver.

## Reproducibility

The notebook removes the previous `delivery_data` directory before execution. This ensures that each run starts from a known state and produces consistent results.

## Limitations

- The demonstration uses a small in-memory dataset.
- Storage is local to the notebook runtime.
- The pipeline is executed in one notebook rather than through an external orchestrator.
- Production deployments would normally use cloud object storage and parameterized configuration.

## Conclusion

This notebook demonstrates a working Delta Lakehouse with Bronze, Silver, and Gold layers, a real business-key `MERGE`, schema enforcement, and an aggregated Gold table.
